In [2]:
import chromadb
import os
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from chromadb.api.types import EmbeddingFunction


c:\Users\suraj\LLM_projects\RAG_memory\LLM_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# -------- Read docs from text file --------
FILE_PATH = "rag_documents.txt"

def load_documents(file_path=FILE_PATH):
    if not os.path.exists(file_path):
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        docs = [line.strip() for line in f.readlines() if line.strip()]
    return docs


def save_response_to_docs(response, file_path=FILE_PATH):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(response + "\n")


In [4]:

documents = load_documents()
ids = [str(i) for i in range(len(documents))]


In [5]:
# -------- Embedding Function Wrapper --------
class MiniLMEmbeddings(EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def __call__(self, texts):
        return self.model.encode(texts).tolist()

    def name(self):
        return "all-MiniLM-L6-v2"


embedding_function = MiniLMEmbeddings()

In [6]:
# -------- Chroma Setup --------
client = chromadb.PersistentClient(path="./rag_db")

# Reset for clean run
try:
    client.delete_collection("rag_collection")
except:
    pass

collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)

# Add docs read from file
if documents:
    collection.add(
        documents=documents,
        ids=ids
    )


In [7]:
# -------- Load an Open (Non-Gated) Model --------
# ✅ Replace Gemma due to access restrictions
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)
llm2 = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Device set to use cuda:0


In [8]:

# -- function to reduce the size of the RAG document -- 
def enforce_doc_size_limit(max_chars=10000, summary_chars=5000):
    if not os.path.exists(FILE_PATH):
        return

    with open(FILE_PATH, "r", encoding="utf-8") as f:
        content = f.read()

    if len(content) <= max_chars:
        return  # ✅ Safe — do nothing

    print("⚠️ Document too large — summarizing and replacing content...")

    # Summarize the entire document via LLM
    prompt = f"""
Summarize the following conversation history into no more than {summary_chars} characters 
while preserving key information for future context:

{content}

Summary:
    """

    summary = llm2(prompt, max_new_tokens=summary_chars, do_sample=True)[0]["generated_text"].strip()

    # ✅ Replace full file with summary only
    with open(FILE_PATH, "w", encoding="utf-8") as f:
        f.write(summary + "\n")

    # Rebuild ChromaDB fully with new content
    new_docs = load_documents()
    new_ids = [str(i) for i in range(len(new_docs))]

    client.delete_collection("rag_collection")
    new_collection = client.get_or_create_collection(
        name="rag_collection",
        embedding_function=embedding_function
    )
    new_collection.add(documents=new_docs, ids=new_ids)

    print("✅ Replacement summary stored and ChromaDB rebuilt.")


In [ ]:
# -------- RAG Query Function --------
def rag_query(question, k=3, store_response=True):
    results = collection.query(query_texts=[question], n_results=k)
    context = "\n".join(results["documents"][0])

    prompt = f"""
Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:
{context}

Question: {question}
Answer:
    """

    answer = llm(prompt, max_new_tokens=150, do_sample=True)[0]["generated_text"].strip()

    if store_response:
        prompt2 = f"""
Summarise the text after the colons in a format of a discussion with a question and an answer: {answer}"""

        Store_answer = llm2(prompt2, max_new_tokens=150, do_sample=True)[0]["generated_text"].strip()
        save_response_to_docs(Store_answer)
        enforce_doc_size_limit()
    return answer



In [20]:
# -------- Test Query --------
print("\nRAG Response:")
print(rag_query("Just give me the answer to the following question: who was Einstein?"))



RAG Response:
Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:
Would you like to continue this discussion? Please feel free to ask follow-up questions or share other aspects of Einstein's life and work that interest you.
Einstein's most famous work came in 1905 when he published what became known as the Special Theory of Relativity. This theory introduced the concepts of time dilation, length contraction, and the equivalence of mass and energy, which are now fundamental principles in modern physics. His later theories, such as the General Theory of Relativity, expanded these ideas to include gravity as curvature of spacetime caused by mass or energy.
Einstein's intellectual contributions extended beyond physics; he also made significant advancements in mathematics, philosophy, and cosmology. His legac

In [ ]:
llm_output = llm("Just give me the answer to the following question: who was Einstein?", max_new_tokens=150, do_sample=False)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [27]:
answer = llm_output.replace("Just give me the answer to the following question: who was Einstein?", "").strip()


AttributeError: 'list' object has no attribute 'replace'

In [29]:
prompt = "Just give me the answer to the following question: who was Einstein?"
raw_output = llm(prompt, max_new_tokens=150, do_sample=True)[0]["generated_text"]
# Remove the prompt part
answer = raw_output.replace(prompt, "").strip()


In [30]:
answer

'The famous physicist Albert Einstein is known for his contributions to physics, particularly in the development of quantum theory and relativity. He is also recognized for his formulation of the famous equation E=mc², which relates mass and energy.\n\nEinstein received numerous awards and honors throughout his life, including the Nobel Prize in Physics in 1921 "for his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect." His work on general relativity revolutionized our understanding of gravity as a geometric property of space-time rather than a force between masses.\n\nSome key points about Albert Einstein:\n\n- Born: March 14, 1879\n- Died: April 18, 195'